# **Projeto M1.2 – Operações Pontuais e Filtragem Espacial**

**Universidade do Vale do Itajaí**


*   **Curso:** Ciência da Computação
*   **Disciplina:** Processamento de Imagens
*   **Professor:** Felipe Viel

**Alunos:**


*   Ismael Antonio da Silva Junior
*   Lucas Lopes Baroni


##
---

## **Instruções**





1.   Para prosseguir com a correção do trabalho, certifique-se de baixar as três imagens presentes na pasta `imagens` (`ISIC_0010925.jpg`, `ISIC_0011094.jpg`, `ISIC_0001119.jpg`) no repositório no qual esse trabalho foi entregue;

2.   Depois de baixá-las, arraste-as para a aba **arquivos** desse ambiente colab. Dessa forma, essas imagens estarão acessíveis durante a sessão para o acesso via código. Lembre-se de repetir o processo caso inicie um novo ambiente!

3. Recomenda-se que o código das células sejam executados sequencialmente. Isso porque existe uma reutilização de certas variáveis e funções previamente utilizadas. Se você executar acidentalmente fora da ordem, recomendo que exclua a sessão e inicie uma nova.

##

---

## **Básico**

In [ ]:
import os

import math
import numpy as np

import cv2
from google.colab.patches import cv2_imshow

import matplotlib.pyplot as plt

from skimage.util import random_noise
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity

ARQUIVOS_DATASET = "/content/"

In [ ]:
def imagem_colorida_para_cinza(img):
  if img is None:
    raise ValueError("img deve ser uma imagem colorida.")

  return np.array((0.114 * img[:, :, 0] + 0.587 * img[:, :, 1] + 0.299 * img[:, :, 2]), dtype=np.uint8)

def calcular_histograma(img_cinza, normalizado=False):
  if img_cinza.ndim != 2:
        raise ValueError("A imagem deve estar em escala de cinza.")

  histograma = np.zeros(256)
  altura, largura = img_cinza.shape

  for i in range(altura):
    for j in range(largura):
      histograma[img_cinza[i,j]] += 1

  if normalizado:
    return histograma / img_cinza.size
  else:
    return histograma

def criar_grafico_histograma(histograma, linhas=False):
    x = np.arange(256)

    fig, ax = plt.subplots(figsize=(10, 4))

    ax.set_xlabel("Intensidade")
    ax.set_ylabel("Frequência")
    ax.set_xlim(0, 255)

    if linhas:
        ax.plot(x, histograma)
    else:
        ax.bar(x, histograma, width=1)

    plt.show()
    plt.close(fig)

def exibir_imagem_histograma(img, histograma, nome_imagem="Imagem", linhas=False):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].imshow(img, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title(nome_imagem)
    axes[0].axis("off")

    x = np.arange(256)

    axes[1].set_title("Histograma")
    axes[1].set_xlabel("Intensidade")
    axes[1].set_ylabel("Frequência")
    axes[1].set_xlim(0, 255)

    if linhas:
        axes[1].plot(x, histograma)
    else:
        axes[1].bar(x, histograma, width=1)

    plt.tight_layout()
    plt.show()
    plt.close(fig)

def metricas_performance(reference, result):
    if reference is None or result is None:
        raise ValueError("Os parâmetros devem estar preenchidos.")

    mse_val = mean_squared_error(reference, result)

    rmse_val = np.sqrt(mse_val)

    psnr_val = peak_signal_noise_ratio(
        reference,
        result,
        data_range=255
    )

    ssim_val = structural_similarity(
        reference,
        result,
        data_range=255,
        channel_axis=None
    )

    return {
        "mse": mse_val,
        "rmse": rmse_val,
        "psnr": psnr_val,
        "ssim": ssim_val
    }

##
---

## **1. Seleção das Imagens**

Escolher três melanomas de uma mesma região anatômica do conjunto ISIC e justificar a seleção.

Antes de escolher as imagens, é fundamental estabelecer o objetivo do pipeline a ser desenvolvido. Isso porque ele orientará os critérios que devem ser utilizados para selecionar as capturas adequadas para testes.

Por isso, visando consolidar um pipeline genêrico para lidar com o tratamento de diferentes melanomas, nosso grupo escolheu o seguinte objetivo:

- **Objetivo**: Modelar um conjunto de etapas sequenciais de processamento de imagens que viabilizem a correção e realce de formas e detalhes de melanomas com distintos contrastes e qualidades de captura.

Com esse objetivo em mãos, para avaliar se o trabalho desenvolvido atende as necessidades estabelecidas por ele, precisamos selecionar imagens que fazem parte do escopo que ele promete melhorar. Por isso, os critérios a serem utilizados para nortear a avaliação do desempenho do pipeline baseia-se na escolha de imagens com:

*   Variação do contraste do melanoma em relação à pele;
*   Presença ou ausência de artefatos visuais que podem prejudicar a análise da imagem.

Nesse contexto, foram escolhidas as seguintes imagens da base de dados ISIC:

*   `ISIC_0010925.jpg`
*   `ISIC_0011094.jpg`
*   `ISIC_0001119.jpg`

A apresentação das imagens e sua devida caracterização serão desenvolvidas no próximo tópico. Ademais, será possível compreender melhor porque as imagens possuem características desejáveis para os critérios que estabelecemos.

In [ ]:
imgs_originais = ['ISIC_0010925.jpg', 'ISIC_0011094.jpg', 'ISIC_0001119.jpg']

for index, nome_imagem in enumerate(imgs_originais):
  print(f'\nImagem {index + 1} - {nome_imagem}\n')
  cv2_imshow(cv2.imread(ARQUIVOS_DATASET + nome_imagem))

##
---

## **2. Caracterização inicial**

Apresentar as imagens originais e identificar seus principais problemas de qualidade.

In [ ]:
imgs_originais = ['ISIC_0010925.jpg', 'ISIC_0011094.jpg', 'ISIC_0001119.jpg']

for index, nome_imagem in enumerate(imgs_originais):
  img = cv2.imread(ARQUIVOS_DATASET + nome_imagem)
  img_cinza = imagem_colorida_para_cinza(img)

  print(f'\nImagem {index + 1} - {nome_imagem}\n')
  cv2_imshow(img)
  exibir_imagem_histograma(img_cinza, calcular_histograma(img_cinza, normalizado=True), nome_imagem=nome_imagem, linhas=True)



*   **Imagem 1 - ISIC_0010925.jpg:** Ela apresenta um melanoma presente em uma região bem delimitada, com cores e bordas que se destacam da tonalidade da pele (bom contraste). O principal problema de qualidade presente nela são as linhas horizontais presente na parte direita da figura, referente a lente utilizada na captura da fotografia. Apesar disso, como essas linhas estão em uma região que não possui partes do melanoma, ele tende a não prejuicar processamentos futuros e análises. Ela foi escolhida para os testes principalmente pelo fato de apresentar um melanoma bem definido e com bom contraste.

*   **Imagem 2 - ISIC_0011094.jpg:** Ela apresenta um melanoma bem disperso, com cores e bordas fracas em relação a tonalidade da pele (baixo contraste). Como o melanoma é mais "discreto" quando comparado com os demais, um dos problemas de qualidade da imagem é justamente a delimitação clara das características da lesão. Ademais, existe um pequeno desfoque das informações mais próximas da borda. Ela foi escolhida para os testes principalmente pelo fato de apresentar um melanoma pouco definido, disperos e com baixo contraste.

*   **Imagem 3 - ISIC_0001119.jpg:** Ela apresenta um melanoma presente em uma região bem delimitada da pele, com cores e bordas intensas em relação a tonalidade da pele (bom contraste). Apesar disso, a imagem possui inúmeros problemas de qualidade que podem atrapalhar a análise clínica do caso, dentre eles temos: excesso de fios de cabelo na região, data e horário da captura no canto inferior direito, linhas diagonais no canto inferior esquerdo da imagem, referente a lente utilizada na captura da fotografia. Ela foi escolhida para os testes devido aos diversos artefatos presentes.

Com esse conjunto de imagens, torna-se possível consolidar testes robustos levando em consideração variações significativas na intensidade, contraste, distribuição espacial e artefatos ruidosos.

##
---

## **3. Inserção de ruído**

Adicionar ruído controlado às imagens e registrar os parâmetros utilizados.

In [ ]:
def adicionar_ruido(img_cinza, tipo, variancia, semente):
    img_noise = random_noise(img_cinza, mode=tipo, var=variancia, rng=semente)
    return (img_noise * 255).astype(np.uint8)

In [ ]:
imgs_originais = ['ISIC_0010925.jpg', 'ISIC_0011094.jpg', 'ISIC_0001119.jpg']

# Armazenará os resultados do processamento do pipeline em relação à uma imagem base sem ruído.
imgs_referencia = {}

# Armazenará os resultados do processamento do pipeline em relação à uma imagem base com ruído.
imgs_resultado = {}

VARIANCIA = 0.003

# Semente para manter o mesmo padrão de ruído em múltiplas execuções
SEMENTES = [42, 43, 44]

for index, nome_imagem in enumerate(imgs_originais):
    img_temp = cv2.imread(ARQUIVOS_DATASET + nome_imagem)

    img_cinza = imagem_colorida_para_cinza(img_temp)

    # Imagem sem ruído
    imgs_referencia[nome_imagem] = img_cinza

    # Imagem com ruído
    imgs_resultado[nome_imagem] = adicionar_ruido(img_cinza, 'gaussian', VARIANCIA, SEMENTES[index])

    print(f'\nImagem {index + 1} - {nome_imagem}:\n')

    print('\nSem ruído | Ruído gaussiano:\n')
    cv2_imshow(np.hstack([imgs_referencia[nome_imagem], imgs_resultado[nome_imagem]]))

    print('\nSem ruído:\n')
    exibir_imagem_histograma(imgs_referencia[nome_imagem], calcular_histograma(imgs_referencia[nome_imagem], normalizado=True), nome_imagem=(nome_imagem + ' (sem ruído)'), linhas=True)

    print('\nRuído gaussiano:\n')
    exibir_imagem_histograma(imgs_resultado[nome_imagem], calcular_histograma(imgs_resultado[nome_imagem], normalizado=True), nome_imagem=(nome_imagem + ' (com ruído gaussiano)'), linhas=True)



Escolhemos o tipo de ruído *Gaussiano*, isso porque ele representa de forma simplificada perturbações aleatórias associadas principalmente aos componentes eletrônicos e térmicos do sistema de captura, sendo o caso mais adequadado no cenário proposto.

A captura por sensores costuma ter uma combinação de:
- ruído de fótons, normalmente modelado por Poisson;
- ruído eletrônico/de leitura e térmico, frequentemente aproximado como ruído Gaussiano aditivo.

Essa conclusão é sustentada por Foi et al. (2008), no artigo *"Practical Poissonian-Gaussian Noise Modeling and Fitting for Single-Image Raw-Data"*, que propõem, para sensores digitais, um modelo de ruído Poisson–Gaussiano: o componente Poisson representa a detecção de fótons, enquanto o componente Gaussiano modela as perturbações estacionárias restantes do sensor, como ruídos eletrônicos e térmicos.

Convertemos as imagens coloridas para tons de cinza e aplicamos a função ` random_noise ` da biblioteca `skimage` para adicionar ruído controlado.

`random_noise(img_cinza, mode='gaussian', var=0.003)`

- `img_cinza`: imagem de entrada em escala de cinza.

- `mode='gaussian'`: define o método de ruído aleatório, no nosso caso, definimos o Gaussiano pelos motivos explicados acima.

- `var=0.003`: variância do ruído Gaussiano, isto é, controla sua intensidade. Esse valor corresponde a um desvio padrão de aproximadamente 0,0548 na escala normalizada \([0,1]\). Dessa forma, foi introduzida uma perturbação visualmente perceptível, mas que ainda preserva as estruturas gerais da lesão e permite avaliar de maneira controlada a capacidade dos filtros de suavização e realce. Definimos esse valor como moderado.

##
---

## **4. Operação pontual**

Aplicar uma técnica de transformação de intensidade escolhida pelo grupo.

In [ ]:
def equalizar_histograma(img_cinza):
    if img_cinza.ndim != 2:
        raise ValueError("A imagem deve estar em escala de cinza.")

    histograma = calcular_histograma(img_cinza, normalizado=True)

    altura, largura = img_cinza.shape
    acumulado = np.zeros(256)
    soma = 0.0

    for intensidade in range(256):
        soma += histograma[intensidade]
        acumulado[intensidade] = soma

    mapeamento = np.zeros(256, dtype=np.uint8)

    for intensidade in range(256):
        nova_intensidade = 255 * acumulado[intensidade]
        nova_intensidade = int(nova_intensidade + 0.5)

        if nova_intensidade < 0:
            nova_intensidade = 0
        elif nova_intensidade > 255:
            nova_intensidade = 255

        mapeamento[intensidade] = nova_intensidade

    resultado = np.zeros((altura, largura), dtype=np.uint8)

    for i in range(altura):
        for j in range(largura):
            intensidade_original = img_cinza[i, j]
            resultado[i, j] = mapeamento[intensidade_original]

    return resultado


def correcao_gama(img_cinza, gamma):
    if gamma <= 0:
        raise ValueError("O valor de gamma deve ser maior que zero.")

    if img_cinza.ndim != 2:
        raise ValueError("A imagem deve estar em escala de cinza.")

    altura, largura = img_cinza.shape
    imagem_corrigida = np.zeros((altura, largura), dtype=np.uint8)

    for y in range(altura):
        for x in range(largura):
            intensidade = img_cinza[y, x]
            intensidade_normalizada = intensidade / 255.0
            nova_intensidade = 255 * (intensidade_normalizada ** gamma)
            nova_intensidade = int(nova_intensidade + 0.5)

            if nova_intensidade < 0:
                nova_intensidade = 0
            elif nova_intensidade > 255:
                nova_intensidade = 255

            imagem_corrigida[y, x] = nova_intensidade

    return imagem_corrigida

Acima, estão presentes as funções principais e auxiliares para viabilizar a implementação de dois algoritmos de operação pontual de intensidade:


*   Equalização do histograma
*   Correção gamma

Logo abaixo, estudaremos o histograma normalizado das imagens para entendermos inicialmente, do ponto de vista teórico, qual é o método mais adequado de intensidade para esse contexto.


In [ ]:
for index, nome_imagem in enumerate(imgs_resultado):
    img = imgs_resultado[nome_imagem]

    histograma = calcular_histograma(img, normalizado=True)

    print(f'\nImagem {index + 1} - {nome_imagem}:\n')
    exibir_imagem_histograma(img, histograma, nome_imagem=nome_imagem, linhas=True)

- `ISIC_0010925.jpg`: a lesão já apresenta boa diferenciação visual em relação à pele. Seu histograma possui uma distribuição mais ampla, com maior concentração nas intensidades médias e claras.

- `ISIC_0011094.jpg`: observa-se o menor contraste visual entre lesão e pele, pois a lesão é espalhada e apresenta intensidades próximas às do fundo. O histograma está fortemente concentrado em uma faixa de tons claros. Por isso, essa é a imagem que tende a se beneficiar mais da **equalização de histograma**.  

- `ISIC_0001119.jpg`: a lesão escura é perceptível, porém a presença de pelos introduz muitos detalhes escuros e interfere na composição da imagem. O histograma também se concentra principalmente em tons médios, com uma cauda em direção às baixas intensidades, associada tanto à lesão quanto aos pelos. A **equalização de histograma** pode aumentar a visibilidade das variações de intensidade, mas também realça os pelos e a textura da pele.  

A partir da análise que, de modo geral, os histogramas das três imagens não utilizam de maneira uniforme toda a faixa de níveis de cinza. Por isso, provavelmente a **equalização de histograma** terá um desempenho melhor, visto que ela redistribui os níveis de intensidade e aumenta o contraste global de forma adaptada à distribuição de cada imagem. É exatamente isso que desejamos, já que queremos aumentar o contraste entre as estruturas para realçar as partes do melanoma.

Um dos pontos que também reforçam que a equalização do histograma pode ser o método mais adequado é que a correção gama exigiria a escolha de um valor específico de γ e um mesmo parâmetro poderia não produzir resultados adequados para todas as imagens.

De qualquer forma, antes de consolidar qualquer conclusão, foram feitos testes com cada um dos métodos abaixo!

In [ ]:
imgs_op_testes = {}

GAMMA = 1.3

print('\n[OPERAÇÃO PONTUAL] TESTES\n')

for index, nome_imagem in enumerate(imgs_resultado):
  imgs_op_testes[nome_imagem] = {
      'equalizacao_histograma': equalizar_histograma(imgs_resultado[nome_imagem]),
      'gamma': correcao_gama(imgs_resultado[nome_imagem], GAMMA)
  }

  print(f'\nImagem {index + 1} - {nome_imagem}:\n')
  print(f'\nEQUALIZAÇÃO DE HISTOGRAMA:\n')
  histograma = calcular_histograma(imgs_op_testes[nome_imagem]['equalizacao_histograma'], normalizado=True)
  exibir_imagem_histograma(imgs_op_testes[nome_imagem]['equalizacao_histograma'], histograma, nome_imagem=nome_imagem, linhas=True)

  print(f'\nCORREÇÃO GAMMA:\n')
  histograma = calcular_histograma(imgs_op_testes[nome_imagem]['gamma'], normalizado=True)
  exibir_imagem_histograma(imgs_op_testes[nome_imagem]['gamma'], histograma, nome_imagem=nome_imagem, linhas=True)

Foram aplicadas a correção gama, utilizando `γ = 1,3`, e a equalização de histograma às mesmas imagens, a fim de comparar seus efeitos sobre a distribuição de intensidades e o contraste visual.

A correção gama produziu um escurecimento moderado das imagens, porém seus histogramas permaneceram concentrados principalmente em faixas intermediárias de intensidade. Dessa forma, a técnica alterou o brilho global, mas promoveu ganho limitado de contraste entre a lesão e a pele ao redor. Foi utilizado o valor 1.3 para γ por representar um escurecimento moderado. O mesmo parâmetro foi aplicado a todas as imagens para manter a comparação com a equalização de histograma consistente e evitar ajustes específicos que favorecessem individualmente alguma imagem.

A equalização de histograma redistribuiu os níveis de cinza por uma faixa mais ampla de intensidades. Esse comportamento é observado nos histogramas resultantes e reflete-se visualmente em maior separação entre regiões escuras, médias e claras.

- Na imagem `ISIC_0011094.jpg`, onde a lesão é pouco definida, a técnica tornou as variações de intensidade mais perceptíveis.

- Na `ISIC_0010925.jpg`, houve melhor evidência de detalhes internos e das bordas da lesão.

- Já na `ISIC_0001119.jpg`, embora também realce os pelos e a textura da pele, a equalização aumenta a distinção visual da lesão em relação ao fundo.

Portanto, para este conjunto de imagens, a **equalização de histograma** foi considerada mais adequada. A técnica apresentou maior capacidade de aumentar o contraste global de forma adaptada à distribuição específica de cada imagem, enquanto a correção gama depende de um único parâmetro e mostrou-se menos eficaz diante das diferenças de iluminação e contraste presentes no conjunto analisado.

In [ ]:
print('\n [OPERAÇÃO PONTUAL] RESULTADO - EQUALIZAÇÃO DE HISTOGRAMA\n')

for index, nome_imagem in enumerate(imgs_resultado):
    imgs_referencia[nome_imagem] = equalizar_histograma(imgs_referencia[nome_imagem])
    imgs_resultado[nome_imagem] = imgs_op_testes[nome_imagem]['equalizacao_histograma']

    print(f'\nImagem {index + 1} - {nome_imagem}:\n')
    cv2_imshow(imgs_resultado[nome_imagem])

##
---

## **5. Suavização**

Reduzir o ruído utilizando uma técnica de filtragem espacial.

In [ ]:
def estender_bordas_replicando(img_cinza, borda):
    altura, largura = img_cinza.shape

    altura_estendida = altura + 2 * borda
    largura_estendida = largura + 2 * borda

    img_estendida = np.zeros((altura_estendida, largura_estendida), dtype=np.float64)

    for y in range(altura_estendida):
        for x in range(largura_estendida):

            y_original = y - borda
            x_original = x - borda

            if y_original < 0:
                y_original = 0
            elif y_original >= altura:
                y_original = altura - 1

            if x_original < 0:
                x_original = 0
            elif x_original >= largura:
                x_original = largura - 1

            img_estendida[y, x] = img_cinza[y_original, x_original]

    return img_estendida


def arredondar_e_limitar(valor):
    valor = int(valor + 0.5)

    if valor < 0:
        return 0
    elif valor > 255:
        return 255

    return valor


def calcular_convolucao(img_cinza, mascara):
    if img_cinza.ndim != 2:
        raise ValueError("A imagem deve estar em escala de cinza.")

    if mascara.shape[0] != mascara.shape[1] or mascara.shape[0] % 2 == 0:
        raise ValueError("A máscara deve ser quadrada e de dimensão ímpar.")

    altura, largura = img_cinza.shape
    tamanho = mascara.shape[0]
    borda = tamanho // 2

    img_estendida = estender_bordas_replicando(img_cinza, borda)
    resultado = np.zeros((altura, largura), dtype=np.float64)

    for y in range(altura):
        for x in range(largura):
            soma = 0.0

            for i in range(tamanho):
                for j in range(tamanho):
                    pixel = img_estendida[y + i, x + j]
                    peso = mascara[i, j]

                    soma += pixel * peso

            resultado[y, x] = soma

    return resultado


def aplicar_convolucao(img_cinza, mascara):
    resultado_float = calcular_convolucao(img_cinza, mascara)

    altura, largura = resultado_float.shape
    resultado = np.zeros((altura, largura), dtype=np.uint8)

    for y in range(altura):
        for x in range(largura):
            resultado[y, x] = arredondar_e_limitar(resultado_float[y, x])

    return resultado


def criar_mascara_media(tamanho=3):
    if tamanho < 3 or tamanho % 2 == 0:
        raise ValueError("O tamanho da máscara deve ser ímpar e maior ou igual a 3.")

    mascara = np.zeros((tamanho, tamanho), dtype=np.float64)
    peso = 1.0 / (tamanho * tamanho)

    for y in range(tamanho):
        for x in range(tamanho):
            mascara[y, x] = peso

    return mascara


def criar_mascara_gaussiana(tamanho=3, sigma=1.0):
    if tamanho < 3 or tamanho % 2 == 0:
        raise ValueError("O tamanho da máscara deve ser ímpar e maior ou igual a 3.")

    if sigma <= 0:
        raise ValueError("O valor de sigma deve ser maior que zero.")

    borda = tamanho // 2
    mascara = np.zeros((tamanho, tamanho), dtype=np.float64)
    soma_pesos = 0.0

    for y in range(-borda, borda + 1):
        for x in range(-borda, borda + 1):
            expoente = -(x ** 2 + y ** 2) / (2 * sigma ** 2)
            valor = math.exp(expoente)

            mascara[y + borda, x + borda] = valor
            soma_pesos += valor

    for y in range(tamanho):
        for x in range(tamanho):
            mascara[y, x] = mascara[y, x] / soma_pesos

    return mascara


def filtro_media(img_cinza, tamanho=3):
    mascara = criar_mascara_media(tamanho)
    return aplicar_convolucao(img_cinza, mascara)


def filtro_gaussiano(img_cinza, tamanho=3, sigma=1.0):
    mascara = criar_mascara_gaussiana(tamanho, sigma)
    return aplicar_convolucao(img_cinza, mascara)


def ordenar_lista_manual(valores):
    for i in range(1, len(valores)):
        atual = valores[i]
        j = i - 1

        while j >= 0 and valores[j] > atual:
            valores[j + 1] = valores[j]
            j -= 1

        valores[j + 1] = atual

    return valores


def filtro_mediana(img_cinza, tamanho=3):
    if img_cinza.ndim != 2:
        raise ValueError("A imagem deve estar em escala de cinza.")

    if tamanho < 3 or tamanho % 2 == 0:
        raise ValueError("O tamanho da janela deve ser ímpar e maior ou igual a 3.")

    altura, largura = img_cinza.shape
    borda = tamanho // 2

    img_estendida = estender_bordas_replicando(img_cinza, borda)
    resultado = np.zeros((altura, largura), dtype=np.uint8)

    for y in range(altura):
        for x in range(largura):
            valores = []

            for i in range(tamanho):
                for j in range(tamanho):
                    valores.append(img_estendida[y + i, x + j])

            valores_ordenados = ordenar_lista_manual(valores)

            indice_mediana = len(valores_ordenados) // 2
            resultado[y, x] = valores_ordenados[indice_mediana]

    return resultado

Acima, estão presentes as funcões principais e auxiliares para viabilizar a implementação de dois algoritmos de filtragem espacial:


*   Média
*   Gaussiano
*   Mediana

Logo abaixo, cada um desses métodos serão testados.


In [ ]:
TAMANHO_MASCARA = 3
SIGMA = 1.0

# Aplicando Suavizações

imgs_suavizadas = {}

for index, nome_imagem in enumerate(imgs_resultado):
    img_atual = imgs_resultado[nome_imagem]

    img_media = filtro_media(img_atual, TAMANHO_MASCARA)
    img_gaussiana = filtro_gaussiano(img_atual, TAMANHO_MASCARA, SIGMA)
    img_mediana = filtro_mediana(img_atual, TAMANHO_MASCARA)

    imgs_suavizadas[nome_imagem] = {
        'media': img_media,
        'gaussiano': img_gaussiana,
        'mediana': img_mediana
    }

    print(f'\nImagem {index + 1} - {nome_imagem}\n')
    print('Anterior | Média | Gaussiano | Mediana')
    cv2_imshow(np.hstack([img_atual, img_media, img_gaussiana, img_mediana]))


# Avaliação Resultados

resultados_filtros = {}

for nome_imagem in imgs_suavizadas:

    referencia = imgs_referencia[nome_imagem]

    resultados_filtros[nome_imagem] = {}

    for filtro, imagem_filtrada in imgs_suavizadas[nome_imagem].items():

        estatisticas = metricas_performance(referencia, imagem_filtrada)

        resultados_filtros[nome_imagem][filtro] = estatisticas

        print(f"\n{nome_imagem} - {filtro}")

        print(f"MSE:  {estatisticas['mse']:.4f}")
        print(f"RMSE: {estatisticas['rmse']:.4f}")
        print(f"PSNR: {estatisticas['psnr']:.4f} dB")
        print(f"SSIM: {estatisticas['ssim']:.4f}")

Os filtros foram aplicados com janela 3x3, considerando o pixel central e seus oito vizinhos. Esse tamanho foi escolhido por realizar uma suavização moderada, reduzindo o risco de perda excessiva de detalhes relevantes das lesões.

Para o filtro Gaussiano, foi utilizado `SIGMA = 1.0`, que concentra maior peso nos pixels próximos ao centro da máscara e, assim, reduz o ruído de maneira mais gradual que a média simples.

**Tabela de resumo**

| Filtro | MSE médio | RMSE médio | PSNR médio (dB) | SSIM médio |
|---|---:|---:|---:|---:|
| Média | **497,4682** | **21,8736** | **21,5262** | **0,4930** |
| Gaussiano | 504,2514 | 22,0434 | 21,4494 | 0,4927 |
| Mediana | 510,9310 | 22,1512 | 21,4241 | 0,4317 |


A partir da análise do desempenho dos filtros distintos nas imagens, as métricas (MSE, RMSE, PSNR, SSIM) apontam que os filtros mais adequados são a média e o gaussiano.

O filtro da média foi numericamente melhor em `ISIC_0010925.jpg` e `ISIC_0001119.jpg`, enquanto o gaussiano foi melhor em todas as métricas para `ISIC_0011094.jpg`. Contudo, a diferença geral de SSIM entre média e Gaussiano é de apenas 0,0001, ou seja, praticamente um empate técnico.

Já a mediana foi a menos indicada neste experimento. Ela é eficaz para ruído impulsivo, como sal e pimenta, pois elimina valores isolados extremos. Como o ruído escolhido é gaussiano, ele é distribuído de forma contínua em torno da intensidade original, por isso, a mediana é menos adequada a esse tipo de degradação.

Embora o filtro da média tenha apresentado pequena vantagem numérica no resultado agregado, foi escolhido o filtro **gaussiano** como técnica final. Essa decisão considera o empate prático entre os dois métodos e a maior coerência entre o filtro selecionado e a técnica de ruído escolhida, já que o ruído inserido artificialmente possui distribuição gaussiana.

In [ ]:
TAMANHO_MASCARA = 3
SIGMA = 1.0

print('[SUAVIZAÇÃO] RESULTADO - FILTRO GAUSSIANO')

for index, nome_imagem in enumerate(imgs_resultado):
    imgs_referencia[nome_imagem] = filtro_gaussiano(imgs_referencia[nome_imagem], TAMANHO_MASCARA, SIGMA)
    imgs_resultado[nome_imagem] = imgs_suavizadas[nome_imagem]['gaussiano']

    print(f'\nImagem {index + 1} - {nome_imagem}\n')
    cv2_imshow(imgs_resultado[nome_imagem])

##
---

## **6. Aguçamento**

Realçar detalhes ou estruturas utilizando uma técnica espacial adequada.

In [ ]:
def converter_para_uint8(img_float):
    altura, largura = img_float.shape
    resultado = np.zeros((altura, largura), dtype=np.uint8)

    for y in range(altura):
        for x in range(largura):
            resultado[y, x] = arredondar_e_limitar(img_float[y, x])

    return resultado


def criar_mascara_laplaciana(diagonais=False):
    if diagonais:
        return np.array([[1,  1, 1],
                         [1, -8, 1],
                         [1,  1, 1]], dtype=np.float64)

    return np.array([[0,  1, 0],
                     [1, -4, 1],
                     [0,  1, 0]], dtype=np.float64)


def filtro_laplaciano(img_cinza, c=1.0, diagonais=False):
    if img_cinza.ndim != 2:
        raise ValueError("A imagem deve estar em escala de cinza.")

    laplaciano = calcular_convolucao(img_cinza, criar_mascara_laplaciana(diagonais))

    altura, largura = img_cinza.shape
    resultado = np.zeros((altura, largura), dtype=np.float64)

    # g(x, y) = f(x, y) - c * ∇²f(x, y)
    for y in range(altura):
        for x in range(largura):
            resultado[y, x] = img_cinza[y, x] - c * laplaciano[y, x]

    return converter_para_uint8(resultado)


def filtro_high_boost(img_cinza, k=1.5, tamanho=3, sigma=1.0):
    if img_cinza.ndim != 2:
        raise ValueError("A imagem deve estar em escala de cinza.")

    if k < 1:
        raise ValueError("O valor de k deve ser maior ou igual a 1.")

    img_borrada = calcular_convolucao(img_cinza, criar_mascara_gaussiana(tamanho, sigma))

    altura, largura = img_cinza.shape
    resultado = np.zeros((altura, largura), dtype=np.float64)

    # High-Boost: f(x, y) + k * (f(x, y) - f_borrada(x, y))
    for y in range(altura):
        for x in range(largura):
            mascara_detalhes = img_cinza[y, x] - img_borrada[y, x]
            resultado[y, x] = (img_cinza[y, x] + k * mascara_detalhes)

    return converter_para_uint8(resultado)

Acima, estão presentes as funcões principais e auxiliares para viabilizar a implementação de dois algoritmos de aguçamento:


*   Laplaciano
*   High-Boost

Logo abaixo, cada um desses métodos serão testados.


In [ ]:
# Aplicando Aguçamentos

C_LAPLACIANO = 0.28

K_HIGH_BOOST = 1.0
TAMANHO_MASCARA = 3
SIGMA = 1.0

imgs_agucadas = {}

for index, nome_imagem in enumerate(imgs_resultado):
    img_atual = imgs_resultado[nome_imagem]

    img_laplaciano = filtro_laplaciano(img_atual, C_LAPLACIANO)
    img_high_boost = filtro_high_boost(img_atual, K_HIGH_BOOST, TAMANHO_MASCARA, SIGMA)

    imgs_agucadas[nome_imagem] = {
        'laplaciano': img_laplaciano,
        'high_boost': img_high_boost
    }

    print(f'\nImagem {index + 1} - {nome_imagem}\n')
    print('Suavizada | Laplaciano | High-Boost')
    cv2_imshow(np.hstack([img_atual, img_laplaciano, img_high_boost]))


# Avaliação Resultados

resultados_filtros = {}

for nome_imagem in imgs_agucadas:
    referencia = imgs_referencia[nome_imagem]
    resultados_filtros[nome_imagem] = {}

    for filtro, imagem_filtrada in imgs_agucadas[nome_imagem].items():

        estatisticas = metricas_performance(referencia, imagem_filtrada)
        resultados_filtros[nome_imagem][filtro] = estatisticas

        print(f"\n{nome_imagem} - {filtro}")

        print(f"MSE:  {estatisticas['mse']:.4f}")
        print(f"RMSE: {estatisticas['rmse']:.4f}")
        print(f"PSNR: {estatisticas['psnr']:.4f} dB")
        print(f"SSIM: {estatisticas['ssim']:.4f}")

Para o Laplaciano, foi utilizada a máscara de vizinhança-4 de tamanho 3x3:

$$
\begin{bmatrix}
0 & 1 & 0 \\
1 & -4 & 1 \\
0 & 1 & 0
\end{bmatrix}
$$

Essa máscara responde às variações locais de intensidade e evidencia transições abruptas, como contornos e texturas. Foi adotado c = 0.28 na equação:

$$
g(x,y) = f(x,y) - c \cdot \nabla^2 f(x,y)
$$

O valor reduzido foi escolhido para limitar a contribuição do Laplaciano, pois derivadas de segunda ordem também tendem a amplificar ruídos residuais e podem produzir realce excessivo quando aplicadas com ganho elevado.

No High-Boost, a máscara de detalhes foi obtida pela diferença entre a imagem de entrada e sua versão borrada por um filtro Gaussiano normalizado de tamanho 3x3, com $\sigma = 1{,}0$. Esse tamanho de máscara utiliza uma vizinhança local, sendo suficiente para destacar detalhes finos sem alterar excessivamente estruturas maiores da lesão. O valor $\sigma = 1{,}0$ produz um desfoque moderado e coerente com uma máscara pequena: suaviza variações muito locais para formar a referência de baixa frequência, mas preserva a maior parte das estruturas relevantes.

Foi utilizado k = 1.0, resultando em:

$$
g(x,y) = f(x,y) + 1{,}0 \cdot [f(x,y) - f_{borrada}(x,y)]
$$

Esse valor representa um realce moderado da máscara de detalhes. Como a imagem já havia passado por suavização, não foi necessário utilizar um ganho mais alto, que poderia aumentar novamente o ruído ou criar contornos artificiais. Embora os valores de c e k não sejam diretamente equivalentes, pois Laplaciano e High-Boost possuem formulações diferentes, ambos foram definidos de forma conservadora e aplicados com máscaras 3x3, mantendo coerência na escala espacial da comparação.

O High-Boost apresentou melhor desempenho em todas as imagens e em todas as métricas avaliadas: reduziu MSE e RMSE, ao mesmo tempo em que elevou PSNR e SSIM. Portanto, ele preservou uma representação mais próxima da imagem de referência e manteve melhor a estrutura local após o realce.

Tecnicamente, o High-Boost é mais adequado neste cenário porque realça detalhes por meio de uma máscara obtida da própria imagem suavizada, preservando as informações de baixa frequência enquanto reforça variações locais relevantes. Já o Laplaciano depende diretamente de derivadas de segunda ordem, sendo mais sensível a pequenas oscilações de intensidade e ao ruído residual.

Assim, o **High-Boost** foi selecionado como técnica de aguçamento do resultado final.

In [ ]:
K_HIGH_BOOST = 1.0
TAMANHO_MASCARA = 3
SIGMA = 1.0

print('[AGUÇAMENTO] RESULTADO - HIGH BOOST')

for index, nome_imagem in enumerate(imgs_agucadas):
    imgs_referencia[nome_imagem] = filtro_high_boost(imgs_referencia[nome_imagem], K_HIGH_BOOST, TAMANHO_MASCARA, SIGMA)
    imgs_resultado[nome_imagem] = imgs_agucadas[nome_imagem]['high_boost']

    print(f'\nImagem {index + 1} - {nome_imagem}\n')
    cv2_imshow(imgs_resultado[nome_imagem])

##
---

## **7. Segunda configuração**

Modificar uma técnica, parâmetro ou ordem do pipeline e repetir o processamento.

### **PIPELINE ORIGINAL**

Pipeline original proposto pelo trabalho. Apesar dele ter sido executado anteriormente, aqui temos uma re-execução rápida e direta de todas as etapas dele com os parâmetros definidos ao longo do trabalho.

In [ ]:
# Parâmetros

VARIANCIA = 0.003
SEMENTES = [42, 43, 44]

TAMANHO_MASCARA_SUAVIZACAO = 3
SIGMA_SUAVIZACAO = 1.0

TAMANHO_MASCARA_AGUCAMENTO = 3
SIGMA_AGUCAMENTO = 1.0
K_HIGH_BOOST = 1.0

imgs_originais = ['ISIC_0010925.jpg', 'ISIC_0011094.jpg', 'ISIC_0001119.jpg']
imgs_resultado_original = {}

for index, nome_imagem in enumerate(imgs_originais):
  img = cv2.imread(ARQUIVOS_DATASET + nome_imagem)
  img_cinza = imagem_colorida_para_cinza(img)

  # Imagem Referência

  ref_etapa02 = equalizar_histograma(img_cinza)
  ref_etapa03 = filtro_gaussiano(ref_etapa02, TAMANHO_MASCARA_SUAVIZACAO, SIGMA_SUAVIZACAO)
  ref_etapa04 = filtro_high_boost(ref_etapa03, K_HIGH_BOOST, TAMANHO_MASCARA_AGUCAMENTO, SIGMA_AGUCAMENTO)
  referencia_final_original = ref_etapa04

  # Imagem Resultado

  img_etapa01 = adicionar_ruido(img_cinza, 'gaussian', VARIANCIA, SEMENTES[index])
  img_etapa02 = equalizar_histograma(img_etapa01)
  img_etapa03 = filtro_gaussiano(img_etapa02, TAMANHO_MASCARA_SUAVIZACAO, SIGMA_SUAVIZACAO)
  img_etapa04 = filtro_high_boost(img_etapa03, K_HIGH_BOOST, TAMANHO_MASCARA_AGUCAMENTO, SIGMA_AGUCAMENTO)
  imgs_resultado_original[nome_imagem] = img_etapa04

  # Console

  print(f'\nImagem {index + 1} - {nome_imagem}:\n')

  print('\nDegradada:\n')
  cv2_imshow(img_etapa01)

  print('\nResultado:\n')
  cv2_imshow(imgs_resultado_original[nome_imagem])

  exibir_imagem_histograma(img_cinza, calcular_histograma(img_cinza, normalizado=True), nome_imagem='IMAGEM ORIGINAL', linhas=True)
  exibir_imagem_histograma(img_etapa01, calcular_histograma(img_etapa01, normalizado=True), nome_imagem='IMAGEM DEGRADADA', linhas=True)
  exibir_imagem_histograma(referencia_final_original, calcular_histograma(referencia_final_original, normalizado=True), nome_imagem='IMAGEM REFERÊNCIA', linhas=True)
  exibir_imagem_histograma(imgs_resultado_original[nome_imagem], calcular_histograma(imgs_resultado_original[nome_imagem], normalizado=True), nome_imagem='IMAGEM RESULTADO', linhas=True)

  estatisticas = metricas_performance(img_cinza, referencia_final_original)

  print("\nORIGINAL x REFERÊNCIA\n")
  print(f"PSNR: {estatisticas['psnr']:.4f} dB")
  print(f"SSIM: {estatisticas['ssim']:.4f}")

  estatisticas = metricas_performance(img_cinza, img_etapa01)

  print("\nORIGINAL x DEGRADADA\n")
  print(f"PSNR: {estatisticas['psnr']:.4f} dB")
  print(f"SSIM: {estatisticas['ssim']:.4f}")

  estatisticas = metricas_performance(img_cinza, imgs_resultado_original[nome_imagem])

  print("\nORIGINAL x RESULTADO\n")
  print(f"PSNR: {estatisticas['psnr']:.4f} dB")
  print(f"SSIM: {estatisticas['ssim']:.4f}")

  estatisticas = metricas_performance(referencia_final_original, img_etapa01)

  print("\nREFERÊNCIA x DEGRADADA\n")
  print(f"PSNR: {estatisticas['psnr']:.4f} dB")
  print(f"SSIM: {estatisticas['ssim']:.4f}")

  estatisticas = metricas_performance(referencia_final_original, imgs_resultado_original[nome_imagem])

  print("\nREFERÊNCIA x RESULTADO\n")
  print(f"PSNR: {estatisticas['psnr']:.4f} dB")
  print(f"SSIM: {estatisticas['ssim']:.4f}")

### **PIPELINE 2**

Essa configuração altera a ordem das etapas do pipeline original.

`Ruído (gaussiano) -> Suavização (gaussiano) -> Intensidade (equalização do histograma) -> Aguçamento (high-boost)`


A alteração proposta consiste em aplicar a suavização gaussiana antes da equalização do histograma. Como o ruído gaussiano introduz variações aleatórias de intensidade, equalizar a imagem imediatamente após sua degradação pode ampliar também essas variações, tornando os artefatos mais visíveis.

Ao suavizar primeiro, busca-se atenuar parte do ruído antes do aumento de contraste. Em seguida, a equalização pode realçar de forma mais controlada as estruturas de interesse da imagem, como a região da lesão, enquanto o high-boost aplicado ao final procura recuperar bordas e detalhes que possam ter sido suavizados.

Essa configuração foi considerada para investigar experimentalmente o efeito da ordem das operações no resultado final. Mantendo os mesmos parâmetros e alterando apenas a posição da suavização e da equalização, torna-se possível verificar se a redução prévia do ruído favorece a preservação estrutural e a qualidade visual das imagens processadas.

In [ ]:
# Parâmetros

VARIANCIA = 0.003
SEMENTES = [42, 43, 44]

TAMANHO_MASCARA_SUAVIZACAO = 3
SIGMA_SUAVIZACAO = 1.0

TAMANHO_MASCARA_AGUCAMENTO = 3
SIGMA_AGUCAMENTO = 1.0
K_HIGH_BOOST = 1.0

imgs_resultado_pipeline02 = {}

for index, nome_imagem in enumerate(imgs_originais):
    img = cv2.imread(ARQUIVOS_DATASET + nome_imagem)
    img_cinza = imagem_colorida_para_cinza(img)

    # Imagem Referência

    ref_etapa02 = filtro_gaussiano(img_cinza, TAMANHO_MASCARA_SUAVIZACAO, SIGMA_SUAVIZACAO)
    ref_etapa03 = equalizar_histograma(ref_etapa02)
    referencia_final_pipeline02 = filtro_high_boost(ref_etapa03, K_HIGH_BOOST, TAMANHO_MASCARA_AGUCAMENTO, SIGMA_AGUCAMENTO)

    # Imagem Resultado

    img_etapa01 = adicionar_ruido(img_cinza, 'gaussian', VARIANCIA, SEMENTES[index])
    img_etapa02 = filtro_gaussiano(img_etapa01, TAMANHO_MASCARA_SUAVIZACAO, SIGMA_SUAVIZACAO)
    img_etapa03 = equalizar_histograma(img_etapa02)
    img_etapa04 = filtro_high_boost(img_etapa03, K_HIGH_BOOST, TAMANHO_MASCARA_AGUCAMENTO, SIGMA_AGUCAMENTO)
    imgs_resultado_pipeline02[nome_imagem] = img_etapa04

    # Console

    print(f'\nImagem {index + 1} - {nome_imagem}:\n')

    print('\nDegradada:\n')
    cv2_imshow(img_etapa01)

    print('\nResultado:\n')
    cv2_imshow(imgs_resultado_pipeline02[nome_imagem])

    exibir_imagem_histograma(img_cinza, calcular_histograma(img_cinza, normalizado=True), nome_imagem='IMAGEM ORIGINAL', linhas=True)
    exibir_imagem_histograma(img_etapa01, calcular_histograma(img_etapa01, normalizado=True), nome_imagem='IMAGEM DEGRADADA', linhas=True)
    exibir_imagem_histograma(referencia_final_pipeline02, calcular_histograma(referencia_final_pipeline02, normalizado=True), nome_imagem='IMAGEM REFERÊNCIA', linhas=True)
    exibir_imagem_histograma(imgs_resultado_pipeline02[nome_imagem], calcular_histograma(imgs_resultado_pipeline02[nome_imagem], normalizado=True), nome_imagem='IMAGEM RESULTADO', linhas=True)

    estatisticas = metricas_performance(img_cinza, referencia_final_pipeline02)

    print("\nORIGINAL x REFERÊNCIA\n")
    print(f"PSNR: {estatisticas['psnr']:.4f} dB")
    print(f"SSIM: {estatisticas['ssim']:.4f}")

    estatisticas = metricas_performance(img_cinza, img_etapa01)

    print("\nORIGINAL x DEGRADADA\n")
    print(f"PSNR: {estatisticas['psnr']:.4f} dB")
    print(f"SSIM: {estatisticas['ssim']:.4f}")

    estatisticas = metricas_performance(img_cinza, imgs_resultado_pipeline02[nome_imagem])

    print("\nORIGINAL x RESULTADO\n")
    print(f"PSNR: {estatisticas['psnr']:.4f} dB")
    print(f"SSIM: {estatisticas['ssim']:.4f}")

    estatisticas = metricas_performance(referencia_final_pipeline02, img_etapa01)

    print("\nREFERÊCIA x DEGRADADA\n")
    print(f"PSNR: {estatisticas['psnr']:.4f} dB")
    print(f"SSIM: {estatisticas['ssim']:.4f}")

    estatisticas = metricas_performance(referencia_final_pipeline02, imgs_resultado_pipeline02[nome_imagem])

    print("\nREFERÊNCIA x RESULTADO\n")
    print(f"PSNR: {estatisticas['psnr']:.4f} dB")
    print(f"SSIM: {estatisticas['ssim']:.4f}")


### **PIPELINE 3**

Essa configuração altera a ordem das etapas do pipeline original.

`Intensidade (equalização do histograma) -> Ruído (gaussiano) -> Suavização (gaussiano) -> Aguçamento (high-boost)`

Essa configuração consiste em realizar a operação pontual de intensidade antes da criação do ruído controlado.

Desse modo, o ruído não é intensificado pela equalização do histograma, tornando as etapas posteriores (suavização e aguçamento, respectivamente) mais eficazes em remover os artefatos, maximizar o contraste e realçar os detalhes dos melanomas.

Entretanto, sabe-se que na prática ele não é possível de ser implementado, pois exige que conheçamos a imagem limpa, sem ruídos. Geralmente, processos como esses são aplicados diretamente à imagens sujas.

Apesar disso, isso não descarta a importância dessa configuração como meio de estudo acadêmico / científico, demonstrando eficientemente que quando o aumento da intensidade é aplicado em uma imagem ruidosa os artefatos são intensificados, prejudicando futuros tratamentos.

In [ ]:
# Parâmetros

VARIANCIA = 0.003
SEMENTES = [42, 43, 44]

TAMANHO_MASCARA_SUAVIZACAO = 3
SIGMA_SUAVIZACAO = 1.0

TAMANHO_MASCARA_AGUCAMENTO = 3
SIGMA_AGUCAMENTO = 1.0
K_HIGH_BOOST = 1.0

imgs_originais = ['ISIC_0010925.jpg', 'ISIC_0011094.jpg', 'ISIC_0001119.jpg']
imgs_resultado_pipeline03 = {}

for index, nome_imagem in enumerate(imgs_originais):
  img = cv2.imread(ARQUIVOS_DATASET + nome_imagem)
  img_cinza = imagem_colorida_para_cinza(img)

  # Imagem Referência

  ref_etapa02 = equalizar_histograma(img_cinza)
  ref_etapa03 = filtro_gaussiano(ref_etapa02, TAMANHO_MASCARA_SUAVIZACAO, SIGMA_SUAVIZACAO)
  ref_etapa04 = filtro_high_boost(ref_etapa03, K_HIGH_BOOST, TAMANHO_MASCARA_AGUCAMENTO, SIGMA_AGUCAMENTO)
  referencia_final_pipeline03 = ref_etapa04

  # Imagem Resultado

  img_etapa01 = equalizar_histograma(img_cinza)
  img_etapa02 = adicionar_ruido(img_etapa01, 'gaussian', VARIANCIA, SEMENTES[index])
  img_etapa03 = filtro_gaussiano(img_etapa02, TAMANHO_MASCARA_SUAVIZACAO, SIGMA_SUAVIZACAO)
  img_etapa04 = filtro_high_boost(img_etapa03, K_HIGH_BOOST, TAMANHO_MASCARA_AGUCAMENTO, SIGMA_AGUCAMENTO)
  imgs_resultado_pipeline03[nome_imagem] = img_etapa04

  # Console

  print(f'\nImagem {index + 1} - {nome_imagem}:\n')

  print('\nDegradada:\n')
  cv2_imshow(img_etapa02)

  print('\nResultado:\n')
  cv2_imshow(imgs_resultado_pipeline03[nome_imagem])

  exibir_imagem_histograma(img_cinza, calcular_histograma(img_cinza, normalizado=True), nome_imagem='IMAGEM ORIGINAL', linhas=True)
  exibir_imagem_histograma(img_etapa02, calcular_histograma(img_etapa02, normalizado=True), nome_imagem='IMAGEM DEGRADADA', linhas=True)
  exibir_imagem_histograma(referencia_final_pipeline03, calcular_histograma(referencia_final_pipeline03, normalizado=True), nome_imagem='IMAGEM REFERÊNCIA', linhas=True)
  exibir_imagem_histograma(imgs_resultado_pipeline03[nome_imagem], calcular_histograma(imgs_resultado_pipeline03[nome_imagem], normalizado=True), nome_imagem='IMAGEM RESULTADO', linhas=True)

  estatisticas = metricas_performance(img_cinza, referencia_final_pipeline03)

  print("\nORIGINAL x REFERÊNCIA\n")
  print(f"PSNR: {estatisticas['psnr']:.4f} dB")
  print(f"SSIM: {estatisticas['ssim']:.4f}")

  estatisticas = metricas_performance(img_cinza, img_etapa02)

  print("\nORIGINAL x DEGRADADA\n")
  print(f"PSNR: {estatisticas['psnr']:.4f} dB")
  print(f"SSIM: {estatisticas['ssim']:.4f}")

  estatisticas = metricas_performance(img_cinza, imgs_resultado_pipeline03[nome_imagem])

  print("\nORIGINAL x RESULTADO\n")
  print(f"PSNR: {estatisticas['psnr']:.4f} dB")
  print(f"SSIM: {estatisticas['ssim']:.4f}")

  estatisticas = metricas_performance(referencia_final_pipeline03, img_etapa02)

  print("\nREFERÊNCIA x DEGRADADA\n")
  print(f"PSNR: {estatisticas['psnr']:.4f} dB")
  print(f"SSIM: {estatisticas['ssim']:.4f}")

  estatisticas = metricas_performance(referencia_final_pipeline03, imgs_resultado_pipeline03[nome_imagem])

  print("\nREFERÊNCIA x RESULTADO\n")
  print(f"PSNR: {estatisticas['psnr']:.4f} dB")
  print(f"SSIM: {estatisticas['ssim']:.4f}")

##
---

## **8. Análise por métricas**

Selecionar duas métricas dentre as apresentadas em aula e utilizá-las para comparar as imagens original, degradada e processadas. A escolha das métricas deverá ser justificada.

Foram selecionadas as métricas **PSNR** e **SSIM**.

O **PSNR** permite avaliar quantitativamente a fidelidade global da imagem degradada e processada em relação à imagem original.

O **SSIM** avalia a preservação de estruturas locais, contraste e detalhes visuais.

Essa combinação é adequada ao experimento, pois o objetivo do pipeline é reduzir o ruído e realçar estruturas relevantes sem provocar perda excessiva de informação ou introdução de artefatos.

As métricas MSE e RMSE não foram selecionadas porque ambas medem essencialmente o mesmo aspecto: o erro médio entre as intensidades dos pixels da imagem processada e da imagem original. Embora sejam úteis para quantificar diferenças numéricas, essas métricas não avaliam diretamente a preservação das estruturas, dos contornos e do contraste local da lesão.

No item anterior, as comparações entre as imagens originais, degradas, referência e resultado já forma feitas para o contexto isolado de cada pipeline utlizando como base as duas métricas de desempenho escolhidas (**PSNR** e **SSIM**).

Por isso, abaixo o objetivo será comparar o desempenho entre os pipelines para preparar o terreno para a discussão do trabalho.

As configurações avaliadas foram:

- **Pipeline 01:** Ruído gaussiano → Equalização do histograma → Suavização gaussiana → High-boost;

- **Pipeline 02:** Ruído gaussiano → Suavização gaussiana → Equalização do histograma → High-boost;

- **Pipeline 03:** Equalização do histograma → Ruído gaussiano → Suavização gaussiana → High-boost.

### **COMPARAÇÃO ENTRE PIPELINES**

Em todas as tabelas, os valores estão apresentados no formato **PSNR (dB) / SSIM**.

#### Original × Referência

Essa comparação quantifica a diferença introduzida pelas próprias operações de processamento aplicadas à imagem limpa, sem a influência do ruído controlado.

| Imagem | Pipeline 01 | Pipeline 02 | Pipeline 03 |
|---|---:|---:|---:|
| ISIC_0010925 | 17,0453 / 0,7871 | 16,9928 / 0,7653 | 17,0453 / 0,7871 |
| ISIC_0011094 | 11,1057 / 0,4981 | 10,9294 / 0,4773 | 11,1057 / 0,4981 |
| ISIC_0001119 | 13,6139 / 0,6400 | 13,5325 / 0,6094 | 13,6139 / 0,6400 |
| **Média** | **13,9216 / 0,6417** | **13,8182 / 0,6173** | **13,9216 / 0,6417** |

Os Pipelines 01 e 03 possuem a mesma imagem de referência, pois ambos aplicam a sequência equalização, suavização gaussiana e high-boost à imagem limpa. O Pipeline 02 apresenta valores diferentes porque a suavização é realizada antes da equalização.

#### Original × Degradada

Essa comparação representa o efeito da degradação em relação à imagem original. Nos Pipelines 01 e 02, o ruído é aplicado diretamente à imagem original; no Pipeline 03, ele é aplicado após a equalização.

| Imagem | Pipeline 01 | Pipeline 02 | Pipeline 03 |
|---|---:|---:|---:|
| ISIC_0010925 | 25,2208 / 0,3238 | 25,2208 / 0,3238 | 16,5228 / 0,3039 |
| ISIC_0011094 | 25,2216 / 0,3749 | 25,2216 / 0,3749 | 10,8718 / 0,3010 |
| ISIC_0001119 | 25,2246 / 0,2885 | 25,2246 / 0,2885 | 13,3851 / 0,2570 |
| **Média** | **25,2223 / 0,3291** | **25,2223 / 0,3291** | **13,5932 / 0,2873** |

Como a imagem degradada dos Pipelines 01 e 02 é gerada antes de qualquer alteração na ordem das etapas, seus valores são idênticos. No Pipeline 03, a comparação com a imagem original inclui tanto o efeito da equalização quanto o ruído inserido posteriormente.

#### Original × Resultado

Essa comparação mostra a diferença entre a imagem original e o resultado final de cada configuração.

| Imagem | Pipeline 01 | Pipeline 02 | Pipeline 03 |
|---|---:|---:|---:|
| ISIC_0010925 | 18,2396 / 0,3774 | 16,4151 / 0,3333 | 16,9781 / 0,5805 |
| ISIC_0011094 | 11,9538 / 0,2415 | 10,6335 / 0,1929 | 11,0771 / 0,4320 |
| ISIC_0001119 | 15,6253 / 0,1853 | 12,9624 / 0,1469 | 13,6307 / 0,4855 |
| **Média** | **15,2729 / 0,2681** | **13,3370 / 0,2244** | **13,8953 / 0,4993** |

Esses valores devem ser interpretados considerando que equalização e high-boost alteram intencionalmente as intensidades e bordas da imagem. Por isso, a comparação com a referência processada é a medida mais diretamente relacionada à estabilidade de cada pipeline diante do ruído.

#### Referência × Degradada

Essa comparação representa a diferença entre a imagem de referência limpa e a versão degradada antes da aplicação das etapas de restauração.

| Imagem | Pipeline 01 | Pipeline 02 | Pipeline 03 |
|---|---:|---:|---:|
| ISIC_0010925 | 16,4724 / 0,2820 | 16,4262 / 0,2789 | 25,3015 / 0,4029 |
| ISIC_0011094 | 10,9817 / 0,2845 | 10,8102 / 0,2750 | 23,3973 / 0,6181 |
| ISIC_0001119 | 13,3264 / 0,2432 | 13,2502 / 0,2380 | 25,1861 / 0,4652 |
| **Média** | **13,5935 / 0,2699** | **13,4955 / 0,2640** | **24,6283 / 0,4954** |

#### Referência × Resultado

Essa é a comparação central da restauração, pois confronta o resultado obtido a partir da imagem ruidosa com a imagem limpa processada pela mesma configuração.

| Imagem | Pipeline 01 | Pipeline 02 | Pipeline 03 |
|---|---:|---:|---:|
| ISIC_0010925 | 23,1178 / 0,4765 | 22,9185 / 0,4509 | 31,8667 / 0,7466 |
| ISIC_0011094 | 19,9007 / 0,4944 | 19,6792 / 0,4958 | 31,8745 / 0,8882 |
| ISIC_0001119 | 19,3059 / 0,3153 | 19,2609 / 0,3077 | 31,8869 / 0,7802 |
| **Média** | **20,7748 / 0,4287** | **20,6195 / 0,4181** | **31,8760 / 0,8050** |

Ao comparar as médias de referência × degradada e referência × resultado, observam-se as seguintes variações após o processamento:

| Pipeline | Variação média de PSNR | Variação média de SSIM |
|---|---:|---:|
| Pipeline 01 | +7,1813 dB | +0,1588 |
| Pipeline 02 | +7,1240 dB | +0,1542 |
| Pipeline 03 | +7,2477 dB | +0,3096 |

Entre os Pipelines 01 e 02, a alteração da ordem entre suavização e equalização produziu diferenças pequenas, porém mensuráveis. Na média de referência × resultado, o Pipeline 02 apresentou variação de **-0,1553 dB** em PSNR e **-0,0106** em SSIM em relação ao Pipeline 01. Individualmente, a segunda imagem apresentou SSIM de 0,4958 no Pipeline 02 e 0,4944 no Pipeline 01; nas demais comparações de referência × resultado, os valores diferiram conforme a imagem analisada.

O Pipeline 03 apresenta valores de referência × degradada e referência × resultado em uma condição distinta: a equalização é aplicada antes da inserção do ruído. Em relação ao Pipeline 01, suas médias de referência × resultado diferem em **+11,1012 dB** de PSNR e **+0,3763** de SSIM. Essa diferença deve ser analisada juntamente com a mudança na posição em que a degradação foi aplicada.

##
---

## **9. Discussão**

Identificar qual configuração apresentou o melhor resultado para cada imagem e
discutir situações em que uma técnica apresentou comportamento inadequado.

| Imagem | Pipeline 01 | Pipeline 02 | Pipeline 03 |
|---|---:|---:|---:|
| `ISIC_0010925.jpg` | 23,1178 dB / 0,4765 | 22,9185 dB / 0,4509 | **31,8667 dB / 0,7466** |
| `ISIC_0011094.jpg` | 19,9007 dB / 0,4944 | 19,6792 dB / **0,4958** | **31,8745 dB / 0,8882** |
| `ISIC_0001119.jpg` | 19,3059 dB / 0,3153 | 19,2609 dB / 0,3077 | **31,8869 dB / 0,7802** |
| **Média** | **20,7748 dB / 0,4287** | **20,6195 dB / 0,4181** | **31,8760 dB / 0,8050** |

Considerando exclusivamente os resultados quantitativos, o **Pipeline 03** apresentou os maiores valores de PSNR e SSIM nas três imagens. A configuração realizou a equalização do histograma antes da inserção do ruído gaussiano. Dessa forma, o ruído foi introduzido sobre uma imagem cujo contraste já havia sido ajustado, evitando que a equalização ampliasse posteriormente as variações aleatórias de intensidade. Essa condição produziu maior proximidade entre a imagem final e sua respectiva referência processada.

Na imagem `ISIC_0010925.jpg`, que apresenta uma lesão visualmente mais destacada, o Pipeline 03 alcançou PSNR de 31,8667 dB e SSIM de 0,7466. Entre as configurações em que o ruído é inserido antes do processamento, o Pipeline 01 obteve 23,1178 dB e 0,4765, enquanto o Pipeline 02 apresentou 22,9185 dB e 0,4509. A alteração da posição da suavização no Pipeline 02 reduziu os dois indicadores nessa imagem.

Para a imagem `ISIC_0011094.jpg`, caracterizada por uma lesão mais difusa e de menor contraste, o Pipeline 03 obteve 31,8745 dB de PSNR e 0,8882 de SSIM. Nos Pipelines 01 e 02, os valores de PSNR foram 19,9007 dB e 19,6792 dB, respectivamente. O Pipeline 02 apresentou SSIM ligeiramente maior nessa imagem (0,4958 contra 0,4944), porém a diferença de 0,0014 é muito pequena e deve ser interpretada juntamente com a análise visual e o PSNR.

A imagem `ISIC_0001119.jpg` apresentou os menores valores de SSIM entre as configurações práticas: 0,3153 no Pipeline 01 e 0,3077 no Pipeline 02. Essa imagem contém muitos pelos, que constituem estruturas finas e de alta frequência. Tanto a equalização quanto o high-boost podem realçar essas estruturas juntamente com detalhes relevantes da lesão, enquanto a suavização pode atenuá-las de forma parcial. Esse comportamento torna mais difícil preservar a estrutura da imagem sem introduzir ou reforçar elementos indesejados. No Pipeline 03, os valores foram 31,8869 dB e 0,7802, devido à inserção do ruído após a equalização.

Entre as configurações aplicáveis a uma imagem que já foi adquirida com ruído, o **Pipeline 01** apresentou o melhor equilíbrio quantitativo. Ele obteve PSNR maior que o Pipeline 02 nas três imagens e SSIM maior em duas delas. Sua ordem  permitiu aplicar o aumento de contraste, reduzir parte das variações introduzidas pelo ruído e, por fim, recuperar detalhes com o aguçamento.

O **Pipeline 02** apresentou comportamento menos adequado na maior parte das comparações. Ao aplicar a suavização antes da equalização, parte dos detalhes de alta frequência é reduzida antes do aumento de contraste. Posteriormente, a equalização redistribui as intensidades da imagem suavizada, mas não recupera as estruturas que já foram atenuadas. Além disso, o high-boost aplicado ao final pode realçar variações residuais e bordas que permaneceram após o filtro. Esse efeito foi mais perceptível na ISIC_0010925 e na ISIC_0001119, nas quais o Pipeline 02 apresentou redução simultânea de PSNR e SSIM em relação ao Pipeline 01.

Por sua vez, embora o **Pipeline 03** tenha apresentado os maiores valores nas três imagens, ele representa uma condição controlada e idealizada. Sua execução pressupõe que a equalização seja aplicada à imagem limpa antes da degradação, situação que não ocorre no uso direto de imagens clínicas já adquiridas com ruído. Portanto, seu principal papel no experimento é evidenciar como a aplicação da equalização sobre uma imagem ruidosa pode amplificar artefatos e dificultar as etapas posteriores de restauração.